# 01 — Dataset Preparation and Preprocessing

This notebook prepares the **Benign and Malicious QR Codes** dataset used in:

> **A Dual Representation Framework for Malicious QR Code Detection Using Fused Feature Learning and Deep Visual Modeling**

It preserves the preparation procedure used in the original experiment while replacing Google Colab-specific paths with portable repository-relative paths.

## Operations

1. Configure deterministic execution.
2. Locate or extract the Kaggle dataset.
3. Detect the benign and malicious class directories.
4. Build image-path and label metadata.
5. inspect image dimensions, modes, channels, and readability.
6. Remove missing paths, duplicate paths, missing files, unreadable images, invalid dimensions, and invalid labels.
7. Create stratified training, validation, and test partitions using the reported **70%/15%/15%** protocol.
8. Export metadata files for all downstream notebooks.

## Dataset

**Benign and Malicious QR Codes**  
Kaggle source: `https://www.kaggle.com/datasets/samahsadiq/benign-and-malicious-qr-codes`

The raw dataset is not redistributed with this repository.

## 1. Imports and Reproducibility

The original experiment used seed `42`. The same seed is retained here for Python and NumPy.

In [ ]:
import random
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed: {SEED}")

## 2. Repository Paths

This notebook supports execution from either:

- the repository root; or
- the `Notebooks/` directory.

Expected local structure:

```text
malicious-qr-code-detection/
├── Notebooks/
│   └── 01_dataset_preparation.ipynb
├── Data/
│   ├── raw/
│   └── processed/
├── Results/
│   └── tables/
├── Models/
└── Documentation/
```

In [ ]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "Data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

RESULTS_DIR = PROJECT_ROOT / "Results"
TABLES_DIR = RESULTS_DIR / "tables"

for directory in [RAW_DATA_DIR, PROCESSED_DATA_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root        :", PROJECT_ROOT)
print("Raw dataset folder  :", RAW_DATA_DIR)
print("Processed data folder:", PROCESSED_DATA_DIR)
print("Results table folder:", TABLES_DIR)

## 3. Locate or Extract the Dataset

Place either:

- the extracted Kaggle dataset under `Data/raw/`; or
- its ZIP archive directly inside `Data/raw/`.

Extraction is skipped when the required class directories already exist.

In [ ]:
DATASET_ZIP = None
# Optional explicit example:
# DATASET_ZIP = RAW_DATA_DIR / "Benign and Malicious QR codes.zip"

def contains_required_class_folders(base_dir: Path) -> bool:
    folder_names = {
        path.name.strip().lower()
        for path in base_dir.rglob("*")
        if path.is_dir()
    }
    return {"benign", "malicious"}.issubset(folder_names)

if not contains_required_class_folders(RAW_DATA_DIR):
    if DATASET_ZIP is None:
        zip_candidates = sorted(RAW_DATA_DIR.glob("*.zip"))

        if len(zip_candidates) == 1:
            DATASET_ZIP = zip_candidates[0]
        elif len(zip_candidates) > 1:
            raise RuntimeError(
                "Multiple ZIP archives were found in Data/raw. "
                "Set DATASET_ZIP explicitly to the required archive."
            )

    if DATASET_ZIP is None:
        raise FileNotFoundError(
            "The benign and malicious folders were not found, and no ZIP "
            f"archive was detected in: {RAW_DATA_DIR}"
        )

    DATASET_ZIP = Path(DATASET_ZIP)

    if not DATASET_ZIP.exists():
        raise FileNotFoundError(f"Dataset archive not found: {DATASET_ZIP}")

    with zipfile.ZipFile(DATASET_ZIP, "r") as archive:
        archive.extractall(RAW_DATA_DIR)

    print(f"Dataset extracted from: {DATASET_ZIP}")
else:
    print("The extracted dataset is already available.")

## 4. Detect the Class Directories

Recursive discovery is used because the downloaded archive may include an additional wrapper directory.

In [ ]:
def find_class_folder(base_path: Path, class_name: str) -> Path:
    matches = [
        path
        for path in base_path.rglob("*")
        if path.is_dir() and path.name.strip().lower() == class_name.lower()
    ]

    if not matches:
        raise FileNotFoundError(
            f"Could not find a '{class_name}' folder below {base_path}."
        )

    if len(matches) > 1:
        print(
            f"Warning: multiple '{class_name}' folders were found. "
            f"Using: {matches[0]}"
        )

    return matches[0]

BENIGN_DIR = find_class_folder(RAW_DATA_DIR, "benign")
MALICIOUS_DIR = find_class_folder(RAW_DATA_DIR, "malicious")

print("Benign folder   :", BENIGN_DIR)
print("Malicious folder:", MALICIOUS_DIR)

## 5. Build Image-Path and Label Metadata

Repository-relative paths are stored instead of machine-specific absolute paths. This allows the metadata files to remain valid after cloning or moving the project.

In [ ]:
VALID_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"
}

def collect_images(folder_path: Path, label_name: str) -> list[dict]:
    records = []

    for file_path in sorted(folder_path.rglob("*")):
        if file_path.is_file() and file_path.suffix.lower() in VALID_EXTENSIONS:
            records.append(
                {
                    "image_path": file_path.relative_to(PROJECT_ROOT).as_posix(),
                    "label": label_name,
                }
            )

    return records

records = (
    collect_images(BENIGN_DIR, "benign")
    + collect_images(MALICIOUS_DIR, "malicious")
)

metadata_df = pd.DataFrame(records)

if metadata_df.empty:
    raise RuntimeError("No supported image files were found.")

print("Dataset shape:", metadata_df.shape)
print("\nClass distribution:")
print(metadata_df["label"].value_counts())
print("\nMetadata preview:")
display(metadata_df.head())

## 6. Basic Image Inspection

As in the original experiment, image-level metadata is extracted to confirm:

- width and height consistency;
- image mode;
- number of channels;
- image readability.

This inspection is performed before the final cleaning stage.

In [ ]:
def inspect_image(relative_path: str) -> pd.Series:
    absolute_path = PROJECT_ROOT / relative_path

    try:
        with Image.open(absolute_path) as image:
            width, height = image.size
            mode = image.mode
            channels = len(image.getbands())
            image.load()

        return pd.Series(
            {
                "width": width,
                "height": height,
                "mode": mode,
                "channels": channels,
                "readable": True,
            }
        )

    except Exception:
        return pd.Series(
            {
                "width": np.nan,
                "height": np.nan,
                "mode": None,
                "channels": np.nan,
                "readable": False,
            }
        )

image_features_df = metadata_df.copy()
inspection_df = image_features_df["image_path"].apply(inspect_image)
image_features_df = pd.concat([image_features_df, inspection_df], axis=1)

print("Readable images:")
print(image_features_df["readable"].value_counts(dropna=False))

print("\nImage modes:")
print(image_features_df["mode"].value_counts(dropna=False))

print("\nChannel counts:")
print(image_features_df["channels"].value_counts(dropna=False))

print("\nWidth summary:")
print(image_features_df["width"].describe())

print("\nHeight summary:")
print(image_features_df["height"].describe())

## 7. Save Initial Metadata

These files correspond to the two preliminary metadata exports used in the original workflow.

In [ ]:
PATHS_LABELS_CSV = PROCESSED_DATA_DIR / "qr_dataset_paths_labels.csv"
IMAGE_FEATURES_CSV = PROCESSED_DATA_DIR / "qr_dataset_features.csv"

metadata_df.to_csv(PATHS_LABELS_CSV, index=False)
image_features_df.to_csv(IMAGE_FEATURES_CSV, index=False)

print("Saved:")
print(" -", PATHS_LABELS_CSV)
print(" -", IMAGE_FEATURES_CSV)

## 8. Clean the Dataset Metadata

This stage follows the original preprocessing sequence:

1. Remove missing and empty image paths.
2. Remove duplicate image paths.
3. Remove records whose files do not exist.
4. Remove images that OpenCV cannot read.
5. Remove images with invalid dimensions.
6. Normalize labels and retain only `benign` and `malicious`.

Exact-content and perceptual duplicate audits are intentionally reserved for the separate validation and reliability notebook.

In [ ]:
cleaned_df = metadata_df.copy()

cleaning_report = {
    "initial_rows": len(cleaned_df),
}

# Step 1: missing and empty paths
missing_path_mask = cleaned_df["image_path"].isna()
empty_path_mask = (
    cleaned_df["image_path"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

cleaning_report["missing_paths"] = int(missing_path_mask.sum())
cleaning_report["empty_paths"] = int(empty_path_mask.sum())

cleaned_df = cleaned_df[~(missing_path_mask | empty_path_mask)].copy()
cleaned_df["image_path"] = cleaned_df["image_path"].astype(str).str.strip()

# Step 2: duplicate paths
duplicate_path_mask = cleaned_df.duplicated(
    subset=["image_path"], keep="first"
)
cleaning_report["duplicate_paths"] = int(duplicate_path_mask.sum())
cleaned_df = cleaned_df[~duplicate_path_mask].copy()

# Step 3: file existence
exists_mask = cleaned_df["image_path"].apply(
    lambda path: (PROJECT_ROOT / path).exists()
)
cleaning_report["missing_files"] = int((~exists_mask).sum())
cleaned_df = cleaned_df[exists_mask].copy()

# Step 4: OpenCV readability
def is_readable(relative_path: str) -> bool:
    try:
        image = cv2.imread(str(PROJECT_ROOT / relative_path))
        return image is not None
    except Exception:
        return False

readable_mask = cleaned_df["image_path"].apply(is_readable)
cleaning_report["unreadable_images"] = int((~readable_mask).sum())
cleaned_df = cleaned_df[readable_mask].copy()

# Step 5: valid dimensions
def has_valid_dimensions(relative_path: str) -> bool:
    try:
        image = cv2.imread(str(PROJECT_ROOT / relative_path))

        if image is None:
            return False

        height, width = image.shape[:2]
        return height > 0 and width > 0

    except Exception:
        return False

dimension_mask = cleaned_df["image_path"].apply(has_valid_dimensions)
cleaning_report["invalid_dimensions"] = int((~dimension_mask).sum())
cleaned_df = cleaned_df[dimension_mask].copy()

# Step 6: label normalization
cleaned_df["label"] = (
    cleaned_df["label"]
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_label_mask = cleaned_df["label"].isin(["benign", "malicious"])
cleaning_report["invalid_labels"] = int((~valid_label_mask).sum())
cleaned_df = cleaned_df[valid_label_mask].copy()

cleaned_df = cleaned_df.reset_index(drop=True)
cleaning_report["final_rows"] = len(cleaned_df)

print("Cleaning report:")
for key, value in cleaning_report.items():
    print(f"{key:22s}: {value:,}")

print("\nFinal label distribution:")
print(cleaned_df["label"].value_counts())

## 9. Export Cleaned Metadata

In [ ]:
CLEAN_METADATA_CSV = PROCESSED_DATA_DIR / "cleaned_qr_metadata.csv"
CLEANING_REPORT_CSV = TABLES_DIR / "dataset_cleaning_summary.csv"

cleaned_df.to_csv(CLEAN_METADATA_CSV, index=False)

pd.DataFrame(
    {
        "check": list(cleaning_report.keys()),
        "count": list(cleaning_report.values()),
    }
).to_csv(CLEANING_REPORT_CSV, index=False)

print("Saved:")
print(" -", CLEAN_METADATA_CSV)
print(" -", CLEANING_REPORT_CSV)

## 10. Stratified 70%/15%/15% Split

The same two-stage stratified split used in the original experiment is retained:

- 70% training;
- 15% validation;
- 15% testing.

In [ ]:
train_df, temporary_df = train_test_split(
    cleaned_df,
    test_size=0.30,
    stratify=cleaned_df["label"],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    stratify=temporary_df["label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary_df = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "samples": [len(train_df), len(val_df), len(test_df)],
        "benign": [
            int((train_df["label"] == "benign").sum()),
            int((val_df["label"] == "benign").sum()),
            int((test_df["label"] == "benign").sum()),
        ],
        "malicious": [
            int((train_df["label"] == "malicious").sum()),
            int((val_df["label"] == "malicious").sum()),
            int((test_df["label"] == "malicious").sum()),
        ],
    }
)

split_summary_df["percentage"] = (
    split_summary_df["samples"] / len(cleaned_df) * 100
).round(2)

display(split_summary_df)

## 11. Validate Split Independence

This check confirms that an image path cannot appear in more than one partition.

In [ ]:
train_paths = set(train_df["image_path"])
val_paths = set(val_df["image_path"])
test_paths = set(test_df["image_path"])

overlap_counts = {
    "train_validation": len(train_paths & val_paths),
    "train_test": len(train_paths & test_paths),
    "validation_test": len(val_paths & test_paths),
}

print("Path overlap counts:", overlap_counts)

if any(overlap_counts.values()):
    raise RuntimeError("Image-path overlap was detected between dataset splits.")

print("No image-path overlap was detected.")

## 12. Export the Dataset Partitions

The filenames `train.csv`, `val.csv`, and `test.csv` are preserved from the original experiment so that downstream notebooks can use the same interface.

In [ ]:
TRAIN_CSV = PROCESSED_DATA_DIR / "train.csv"
VAL_CSV = PROCESSED_DATA_DIR / "val.csv"
TEST_CSV = PROCESSED_DATA_DIR / "test.csv"
SPLIT_SUMMARY_CSV = TABLES_DIR / "dataset_split_summary.csv"

train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)
split_summary_df.to_csv(SPLIT_SUMMARY_CSV, index=False)

print("Saved:")
print(" -", TRAIN_CSV)
print(" -", VAL_CSV)
print(" -", TEST_CSV)
print(" -", SPLIT_SUMMARY_CSV)

## 13. Final Reproducibility Summary

In [ ]:
assert len(train_df) + len(val_df) + len(test_df) == len(cleaned_df)
assert set(cleaned_df["label"].unique()) == {"benign", "malicious"}
assert not any(overlap_counts.values())

print("=" * 60)
print("DATASET PREPARATION SUMMARY")
print("=" * 60)
print(f"Total clean images     : {len(cleaned_df):,}")
print(f"Benign images          : {(cleaned_df['label'] == 'benign').sum():,}")
print(f"Malicious images       : {(cleaned_df['label'] == 'malicious').sum():,}")
print("-" * 60)
print(f"Training samples       : {len(train_df):,}")
print(f"Validation samples     : {len(val_df):,}")
print(f"Test samples           : {len(test_df):,}")
print("-" * 60)
print(f"Missing paths removed  : {cleaning_report['missing_paths']:,}")
print(f"Empty paths removed    : {cleaning_report['empty_paths']:,}")
print(f"Duplicate paths removed: {cleaning_report['duplicate_paths']:,}")
print(f"Missing files removed  : {cleaning_report['missing_files']:,}")
print(f"Unreadable images      : {cleaning_report['unreadable_images']:,}")
print(f"Invalid dimensions     : {cleaning_report['invalid_dimensions']:,}")
print(f"Invalid labels removed : {cleaning_report['invalid_labels']:,}")
print("=" * 60)
print("Dataset preparation completed successfully.")